# Chapter 5 — Orbits in axisymmetric galaxies: epicycles and the third integral

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/05_axisymmetric_orbits.ipynb)

**Goal.** Follow stars in the Milky Way potential of Chapter 4 and understand their orbits: what is
conserved, why disk stars oscillate gently about circular orbits, and how the local rotation of the
Galaxy is measured.

**You will learn**
1. the conserved quantities $E$ and $L_z$, and the meridional plane with its effective potential;
2. that most orbits obey a **third integral** of motion, revealed by surfaces of section;
3. the **epicycle approximation**: the frequencies $\kappa$, $\nu$, $\Omega$ of near-circular orbits;
4. the **Oort constants** $A$ and $B$, and how they compare with Gaia measurements.

**Units.** $M_\odot$, kpc, km/s (Chapter 4). Frequencies come out in km/s/kpc; $1$ km/s/kpc
$= 1.023$ Gyr$^{-1}$. Times are in AGAMA's unit, 1 kpc/(km/s) $= 0.978$ Gyr.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object
G = agama.G
TIME_UNIT_MYR = agama.getUnits()['time']           # 977.8 Myr

import os
import numpy as np
import matplotlib.pyplot as plt

data_dir = os.path.join(os.path.dirname(agama.__file__), 'data')
pot = agama.Potential(os.path.join(data_dir, 'MWPotential2014.ini'))   # the model of Chapter 4

def vcirc(pot, R):
    """Midplane circular speed sqrt(-R F_R) at radii R (Chapter 0 §0.4)."""
    R = np.atleast_1d(R)
    return np.sqrt(-R * pot.force(np.column_stack([R, 0 * R, 0 * R]))[:, 0])

def to_cyl(traj):
    """Cylindrical R, z, v_R, v_phi of an (N, 6) trajectory (Chapter 4 §4.1)."""
    x, y, z, vx, vy, vz = traj.T
    R = np.hypot(x, y)
    return R, z, (x * vx + y * vy) / R, (x * vy - y * vx) / R

## 5.1 Conserved quantities and the meridional plane

In an axisymmetric, time-independent potential $\Phi(R, z)$ two quantities are conserved:

- the energy $E = \tfrac12 v^2 + \Phi(R, z)$ (no time dependence);
- the angular momentum about the symmetry axis, $L_z = R\,v_\phi$ (no torque about the $z$-axis,
  because $\Phi$ does not depend on $\phi$).

The total angular momentum $|\mathbf L|$ is *not* conserved — unlike in Chapters 1–3 — so orbits are no
longer confined to a plane. With $v_\phi = L_z/R$ fixed by $L_z$, the motion in the **meridional plane**
$(R, z)$, which rotates with the star, is that of a particle in the effective potential

$$ \Phi_{\rm eff}(R, z) = \Phi(R, z) + \frac{L_z^2}{2R^2}, \qquad
   E = \tfrac12\left(v_R^2 + v_z^2\right) + \Phi_{\rm eff}(R, z). $$

Since $v_R^2 + v_z^2 \ge 0$, the star can only go where $\Phi_{\rm eff} \le E$: the boundary
$\Phi_{\rm eff} = E$ is the **zero-velocity curve**.

We launch a star from the Sun's radius ($R = 8$ kpc) with a Sun-like rotation speed but extra radial
and vertical velocity, like a star of the Galaxy's thick disk.

In [ ]:
R0 = 8.0
vc0 = vcirc(pot, R0)[0]
xv0 = np.array([R0, 0, 0, 40.0, 0.9 * vc0, 50.0])    # (x, y, z, vx = v_R, vy = v_phi, vz) at phi = 0
t, traj = agama.orbit(potential=pot, ic=xv0, time=3.0, trajsize=30001, dtype=float)   # 3 units = 2.9 Gyr

R, z, vR, vphi = to_cyl(traj)
E = pot.potential(traj[:, :3]) + 0.5 * np.sum(traj[:, 3:]**2, axis=1)
Lz = traj[:, 0] * traj[:, 4] - traj[:, 1] * traj[:, 3]     # x v_y - y v_x

def phi_eff(R, z, Lz):
    """Effective potential in the meridional plane; R, z are arrays of the same shape."""
    pts = np.column_stack([np.ravel(R), 0 * np.ravel(R), np.ravel(z)])
    return (pot.potential(pts) + 0.5 * Lz**2 / pts[:, 0]**2).reshape(np.shape(R))

Rg, zg = np.meshgrid(np.linspace(3, 14, 221), np.linspace(-3, 3, 121))
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(traj[:, 0], traj[:, 1], lw=0.3); ax[0].plot(0, 0, 'k+')
ax[0].set_aspect('equal'); ax[0].set_xlabel('$x$ [kpc]'); ax[0].set_ylabel('$y$ [kpc]')
ax[0].set_title('seen from above')
ax[1].plot(R, z, lw=0.3)
ax[1].contour(Rg, zg, phi_eff(Rg, zg, Lz[0]), levels=[E[0]], colors='k')   # zero-velocity curve
ax[1].set_xlabel('$R$ [kpc]'); ax[1].set_ylabel('$z$ [kpc]')
ax[1].set_title('meridional plane; black: zero-velocity curve $\\Phi_{\\rm eff} = E$')
plt.tight_layout(); plt.show()

print(f'relative variation: E {np.ptp(E) / abs(E[0]):.1e},  L_z {np.ptp(Lz) / Lz[0]:.1e}')   # ptp = max - min
print(f'max of Phi_eff - E along the orbit: {np.max(phi_eff(R, z, Lz[0]) - E[0]):.2e} (km/s)^2')
assert np.ptp(E) / abs(E[0]) < 1e-6 and np.ptp(Lz) / Lz[0] < 1e-6
assert np.max(phi_eff(R, z, Lz[0]) - E[0]) < 1e-3 * abs(E[0])

**Figure 5.1.** An orbit of a thick-disk-like star in MWPotential2014, integrated for 2.9 Gyr. Left: seen from above, in $(x, y)$. Right: in the meridional plane $(R, z)$, with the zero-velocity curve $\Phi_{\rm eff} = E$ (black).

In Figure 5.1 the orbit is a rosette seen from above (Chapter 3), and in the meridional plane it fills a box-shaped
region. The star stays inside the zero-velocity curve, as it must — but notice that it does **not**
fill all of it: the corners of the region allowed by $E$ and $L_z$ are never visited.

## 5.2 A third integral and surfaces of section

If $E$ and $L_z$ were the only constraints, the orbit would eventually wander everywhere inside the
zero-velocity curve. That it stays in a smaller region means a **third integral** of motion
$I_3(R, z, v_R, v_z)$ is (at least approximately) conserved. For a few special potentials it is
known exactly (Stäckel potentials, Chapter 9); for the Milky Way it has no simple formula.

A **surface of section** shows this directly. Every time the star crosses the midplane upwards
($z = 0$, $v_z > 0$) we record $(R, v_R)$. If the orbit obeys only $E$ and $L_z$, these points fill an
area; if it obeys a third integral, they lie on a **curve**. We launch several orbits with the same $E$
and $L_z$ from different points of the midplane and overlay their sections.

*Crossing detection:* between two stored steps $i$ and $i+1$ with $z_i < 0 \le z_{i+1}$, the crossing
is at fraction $f = -z_i/(z_{i+1} - z_i)$ of the step; we interpolate $R$ and $v_R$ linearly to it.

In [ ]:
Lz0, E0 = Lz[0], E[0]

# Starting points: on the midplane with v_R = 0, at radii where Phi_eff(R, 0) < E. The leftover energy
# goes into v_z. A star started where Phi_eff(R, 0) = E exactly would have v_z = 0 (a planar orbit).
R_mid = np.linspace(3, 14, 2001)
allowed = R_mid[phi_eff(R_mid, 0 * R_mid, Lz0) < E0]            # boolean mask selects the allowed radii
R_start = np.linspace(allowed.min(), allowed.max(), 9)[1:-1]    # 7 radii, avoiding the end points
vz_start = np.sqrt(2 * (E0 - phi_eff(R_start, 0 * R_start, Lz0)))
ic = np.column_stack([R_start, 0 * R_start, 0 * R_start, 0 * R_start, Lz0 / R_start, vz_start])

orbits = agama.orbit(potential=pot, ic=ic, time=20.0, trajsize=200001, dtype=float)   # ~20 Gyr each

def section(traj):
    """(R, v_R) at upward crossings of z = 0, by linear interpolation between stored steps."""
    R, z, vR, _ = to_cyl(traj)
    i = np.where((z[:-1] < 0) & (z[1:] >= 0))[0]
    f = -z[i] / (z[i + 1] - z[i])
    return R[i] + f * (R[i + 1] - R[i]), vR[i] + f * (vR[i + 1] - vR[i])

plt.figure(figsize=(7, 5))
for k, (t_k, traj_k) in enumerate(orbits):
    Rs, vRs = section(traj_k)
    plt.plot(Rs, vRs, '.', ms=2, label=f'start $R$ = {R_start[k]:.2f}')
plt.xlabel('$R$ [kpc]'); plt.ylabel('$v_R$ [km/s]')
plt.title(f'surface of section $z = 0$, $v_z > 0$;  $E$, $L_z$ the same for all orbits')
plt.legend(fontsize=8, markerscale=4); plt.show()

**Figure 5.2.** Surface of section $z = 0$, $v_z > 0$, in the $(R, v_R)$ plane for seven orbits with the same $E$ and $L_z$ as the orbit of Figure 5.1, started on the midplane at the radii given in the legend.

In Figure 5.2, most orbits trace a closed curve — an **invariant curve** — nested inside the others, around the
nearly fixed point of the innermost orbit (a "shell" orbit that crosses the midplane at the same $R$
every time). These orbits are **regular**: each obeys a third integral, whose value labels its curve.

One orbit is different: it traces a *chain* of separate arcs. It is trapped around a **resonant** orbit,
whose radial and vertical frequencies are in a ratio of small integers, and each arc is a small island
around one of that orbit's crossings. It is still regular — the arcs are curves, not scattered points —
but it belongs to a different orbit family. (**Chaotic** orbits, whose points fill areas of the section,
appear in non-axisymmetric potentials such as bars; see the optional chapter on bars.)

### Is the "vertical energy" the third integral?

A natural guess is the energy of vertical oscillation,
$E_z = \tfrac12 v_z^2 + \Phi(R, z) - \Phi(R, 0)$. It is conserved only if radial and vertical motions
are completely decoupled, which is approximately true for orbits that stay close to the midplane.
AGAMA's `ActionFinder` computes the **vertical action** $J_z$ (Chapter 9), which is a true integral for
regular orbits up to the accuracy of its approximation. We compare the two along one of our orbits.

In [ ]:
af = agama.ActionFinder(pot)
traj_k = orbits[-1, 1]                             # the last orbit of the section
sample = traj_k[::2000]                            # 101 points along the orbit
Rk, zk, _, _ = to_cyl(sample)
Ez = 0.5 * sample[:, 5]**2 + pot.potential(sample[:, :3]) \
     - pot.potential(np.column_stack([Rk, 0 * Rk, 0 * Rk]))
Jz = af(sample)[:, 1]                              # actions are (J_r, J_z, J_phi); column 1 is J_z

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(orbits[-1, 0][::2000] * TIME_UNIT_MYR / 1000, Ez / Ez.mean())
ax[0].set_title('vertical energy $E_z$ / mean'); ax[0].set_xlabel('time [Gyr]')
ax[1].plot(orbits[-1, 0][::2000] * TIME_UNIT_MYR / 1000, Jz / Jz.mean())
ax[1].set_title('vertical action $J_z$ (AGAMA) / mean'); ax[1].set_xlabel('time [Gyr]')
ax[1].set_ylim(ax[0].get_ylim())                   # same vertical scale on both panels
plt.tight_layout(); plt.show()

print(f'scatter along the orbit: E_z {np.std(Ez) / np.mean(Ez):.1%},  J_z {np.std(Jz) / np.mean(Jz):.2%}')
assert np.std(Jz) / np.mean(Jz) < 0.01 and np.std(Ez) / np.mean(Ez) > 0.05

**Figure 5.3.** Along the last orbit of Figure 5.2, over about 20 Gyr: the vertical energy $E_z$ (left) and the vertical action $J_z$ from AGAMA's `ActionFinder` (right), each divided by its mean and plotted on the same vertical scale.

In Figure 5.3, $E_z$ changes by more than 10% as the star moves in and out radially; $J_z$ stays constant to a fraction
of a per cent. The third integral exists, but it is not the vertical energy. Chapter 9 explains how AGAMA
computes $J_z$.

## 5.3 Near-circular orbits: the epicycle approximation

Most disk stars, like the Sun, move on nearly circular orbits. Expanding $\Phi_{\rm eff}$ to second
order around the circular orbit (the **guiding centre**, at radius $R_g$ where $L_z = R_g v_c(R_g)$), the
motion separates into three simple oscillations:

$$ R - R_g \propto \cos(\kappa t), \qquad z \propto \cos(\nu t), \qquad \dot\phi \approx \Omega , $$

with the **circular**, **epicyclic** and **vertical frequencies**

$$ \Omega^2 = \frac{1}{R}\frac{\partial\Phi}{\partial R}, \qquad
   \kappa^2 = \frac{\partial^2\Phi}{\partial R^2} + \frac{3}{R}\frac{\partial\Phi}{\partial R}, \qquad
   \nu^2 = \frac{\partial^2\Phi}{\partial z^2}, $$

all evaluated in the midplane at $R_g$.

*In AGAMA*, `pot.eval(points, acc=True, der=True)` returns the acceleration $-\nabla\Phi$ and its
derivatives $-\partial^2\Phi/\partial x_i\partial x_j$, stored as six columns in the order
$xx, yy, zz, xy, yz, zx$. On the $x$-axis, $x = R$, so column 0 gives $-\partial^2\Phi/\partial R^2$ and
column 2 gives $-\partial^2\Phi/\partial z^2$.

In [ ]:
def epicycle_frequencies(pot, R):
    """Omega, kappa, nu [km/s/kpc] in the midplane at radii R."""
    R = np.atleast_1d(R)
    acc, der = pot.eval(np.column_stack([R, 0 * R, 0 * R]), acc=True, der=True)
    dPhi_dR = -acc[:, 0]
    d2Phi_dR2, d2Phi_dz2 = -der[:, 0], -der[:, 2]
    Omega = np.sqrt(dPhi_dR / R)
    kappa = np.sqrt(d2Phi_dR2 + 3 * dPhi_dR / R)
    nu = np.sqrt(d2Phi_dz2)
    return Omega, kappa, nu

# Check against three potentials where kappa / Omega is known exactly:
#   Kepler (point mass): kappa = Omega;  flat rotation curve: kappa = sqrt(2) Omega;
#   homogeneous sphere (harmonic potential, here the core of a Plummer sphere): kappa = 2 Omega
tests = [('Kepler', agama.Potential(type='Plummer', mass=1e10, scaleRadius=0), 5.0, 1.0),
         ('flat rotation curve', agama.Potential(type='Logarithmic', v0=200, scaleRadius=0), 5.0, np.sqrt(2)),
         ('homogeneous core', agama.Potential(type='Plummer', mass=1e10, scaleRadius=1), 0.001, 2.0)]
for name, p, R_test, expected in tests:
    Om, ka, _ = epicycle_frequencies(p, R_test)
    print(f'{name:20s}: kappa / Omega = {ka[0] / Om[0]:.6f}  (expected {expected:.6f})')
    assert abs(ka[0] / Om[0] - expected) < 1e-5

R = np.linspace(2, 20, 200)
Om, ka, nu = epicycle_frequencies(pot, R)
plt.figure(figsize=(7, 4))
plt.semilogy(R, Om, label=r'$\Omega$ (circular)')
plt.semilogy(R, ka, label=r'$\kappa$ (epicyclic)')
plt.semilogy(R, nu, label=r'$\nu$ (vertical)')
plt.axvline(R0, color='gray', ls=':')
plt.xlabel('$R$ [kpc]'); plt.ylabel('frequency [km/s/kpc]'); plt.legend()
plt.title('MWPotential2014'); plt.show()

Om0, ka0, nu0 = [f[0] for f in epicycle_frequencies(pot, R0)]
print(f'at R = {R0} kpc: Omega = {Om0:.2f}, kappa = {ka0:.2f}, nu = {nu0:.2f} km/s/kpc;  kappa/Omega = {ka0 / Om0:.3f}')
print(f'periods: azimuthal {2 * np.pi / Om0 * TIME_UNIT_MYR:.0f} Myr, radial {2 * np.pi / ka0 * TIME_UNIT_MYR:.0f} Myr, '
      f'vertical {2 * np.pi / nu0 * TIME_UNIT_MYR:.0f} Myr')

**Figure 5.4.** Circular ($\Omega$), epicyclic ($\kappa$) and vertical ($\nu$) frequencies in the midplane of MWPotential2014 against $R$; the dotted line marks $R = 8$ kpc.

At the Sun's radius (Figure 5.4), a star goes round the Galaxy in about 220 Myr, oscillates radially in about
170 Myr and vertically in about 80 Myr; $\kappa/\Omega \approx 1.34$, close to the flat-rotation-curve
value $\sqrt2$. Vertical oscillations are fast because the disk's mass is concentrated near the midplane.

### Testing the epicycle approximation

We launch a star from the circular orbit at $R_0$ with small kicks, $v_R = 5$ km/s and $v_z = 2$ km/s, and
measure the periods of its radial and vertical oscillations from the times at which $R - R_g$ and $z$
cross zero upwards. The guiding radius comes from AGAMA's `Rcirc(L=...)` (Chapter 1 §1.5). We also
compare with AGAMA's exact frequencies from `ActionFinder` (Chapter 3 §3.2), and evaluate the epicycle
frequencies at $R_g$.

In [ ]:
xv_ep = np.array([R0, 0, 0, 5.0, vc0, 2.0])
t_ep, tr_ep = agama.orbit(potential=pot, ic=xv_ep, time=2.0, trajsize=200001, dtype=float)
R_ep, z_ep, _, _ = to_cyl(tr_ep)
Rg = pot.Rcirc(L=R0 * vc0)
Om_g, ka_g, nu_g = [f[0] for f in epicycle_frequencies(pot, Rg)]

def mean_period(t, signal):
    """Mean interval between upward zero crossings of `signal` (linear interpolation)."""
    i = np.where((signal[:-1] < 0) & (signal[1:] >= 0))[0]
    tc = t[i] - signal[i] * (t[i + 1] - t[i]) / (signal[i + 1] - signal[i])
    return np.diff(tc).mean()

kappa_orbit = 2 * np.pi / mean_period(t_ep, R_ep - Rg)
nu_orbit = 2 * np.pi / mean_period(t_ep, z_ep)
_, Om_agama = af(xv_ep, frequencies=True)          # (Omega_r, Omega_z, Omega_phi)

print(f'                 kappa      nu     Omega   [km/s/kpc]')
print(f'epicycle (R_g): {ka_g:7.3f} {nu_g:7.3f} {Om_g:7.3f}')
print(f'from the orbit: {kappa_orbit:7.3f} {nu_orbit:7.3f}')
print(f'AGAMA frequencies: {Om_agama[0]:7.3f} {Om_agama[1]:7.3f} {Om_agama[2]:7.3f}')
assert abs(kappa_orbit / ka_g - 1) < 0.005 and abs(nu_orbit / nu_g - 1) < 0.005
assert np.allclose(Om_agama, [ka_g, nu_g, Om_g], rtol=0.005)

All three agree to better than 0.5%: for a star like the Sun, the epicycle approximation is excellent.

### The epicycle seen from the guiding centre

In a frame rotating with the guiding centre at angular speed $\Omega_g$, the star moves on a small
ellipse — the **epicycle**. Its tangential and radial semi-axes are in the ratio $Y/X = 2\Omega/\kappa$:
about 1.5 at the Sun. The star goes round it in the **retrograde** sense, opposite to the Galaxy's
rotation. In Figure 5.5 the axes point outwards ($R - R_g$) and forwards along the rotation, so the
Galaxy rotates counter-clockwise and the epicycle should be traversed clockwise; we check this with the
sign of the loop's area. We plot the star's position relative to the guiding
centre, $x = R - R_g$ and $y = R_g(\phi - \Omega_g t)$.

In [ ]:
phi_ep = np.unwrap(np.arctan2(tr_ep[:, 1], tr_ep[:, 0]))
xg = R_ep - Rg
yg = Rg * (phi_ep - Om_g * t_ep)
yg -= np.polyval(np.polyfit(t_ep, yg, 1), t_ep)    # remove a tiny residual drift (second-order effect)
sel = t_ep < 2 * np.pi / ka_g                       # one epicycle

plt.figure(figsize=(5, 5))
plt.plot(xg[sel], yg[sel])
plt.plot(xg[0], yg[0], 'o', label='start')
i_arrow = np.argmin(np.abs(t_ep - 0.2 * 2 * np.pi / ka_g))          # a point a fifth of the way round
plt.annotate('', xy=(xg[i_arrow + 50], yg[i_arrow + 50]), xytext=(xg[i_arrow], yg[i_arrow]),
             arrowprops=dict(arrowstyle='->', lw=2))                # arrow showing the direction of motion
plt.plot(0, 0, 'k+', ms=12, label='guiding centre')
plt.gca().set_aspect('equal')
plt.xlabel('$R - R_g$ [kpc]'); plt.ylabel(r'$R_g(\phi - \Omega_g t)$ [kpc]'); plt.legend(loc='lower left', fontsize=8)
plt.title('epicycle'); plt.show()

ratio = np.ptp(yg[sel]) / np.ptp(xg[sel])           # full widths, so their ratio = Y/X
# signed area of the loop (shoelace formula): positive for counter-clockwise motion in this plot
area = 0.5 * np.sum(xg[sel][:-1] * yg[sel][1:] - xg[sel][1:] * yg[sel][:-1])
print(f'axis ratio Y/X = {ratio:.3f};  2 Omega / kappa = {2 * Om_g / ka_g:.3f};  signed area {area:.3f} kpc^2')
assert abs(ratio / (2 * Om_g / ka_g) - 1) < 0.02 and area < 0

**Figure 5.5.** One epicycle: the position of a nearly circular orbit relative to its guiding centre (+), radial offset $R - R_g$ against azimuthal offset $R_g(\phi - \Omega_g t)$; the arrow shows the direction of motion.

## 5.4 The Oort constants

Stars near the Sun share the Galaxy's differential rotation: those inside the solar circle go round
faster in angle, those outside more slowly. Oort (1927) showed that, to first order in distance, this
produces a pattern in the radial velocities and proper motions of nearby stars described by two numbers,

$$ A = \frac12\left(\frac{v_c}{R} - \frac{dv_c}{dR}\right), \qquad
   B = -\frac12\left(\frac{v_c}{R} + \frac{dv_c}{dR}\right), $$

evaluated at the Sun. $A$ measures the **shear** of the flow and $B$ its **vorticity**. They connect to the
frequencies above:

$$ A - B = \Omega, \qquad A + B = -\frac{dv_c}{dR}, \qquad \kappa^2 = -4B(A - B). $$

Bovy (2017) measured them from the proper motions of 300,000 nearby stars in Gaia DR1:
$A = 15.3 \pm 0.4$, $B = -11.9 \pm 0.4$ km/s/kpc. We compute them for the two Milky Way models of
Chapter 4, taking $dv_c/dR$ by a centred finite difference.

In [ ]:
def oort_constants(pot, R, dR=1e-3):
    v = vcirc(pot, R)[0]
    dv_dR = (vcirc(pot, R + dR)[0] - vcirc(pot, R - dR)[0]) / (2 * dR)   # centred difference, error ~ dR^2
    return 0.5 * (v / R - dv_dR), -0.5 * (v / R + dv_dR), dv_dR

mcmillan = agama.Potential(os.path.join(data_dir, 'McMillan17.ini'))
for name, p, Rsun in [('MWPotential2014', pot, 8.0), ('McMillan (2017)', mcmillan, 8.21)]:
    A, B, dv = oort_constants(p, Rsun)
    Om_s, ka_s, _ = [f[0] for f in epicycle_frequencies(p, Rsun)]
    print(f'{name:16s} at R = {Rsun}: A = {A:5.2f}, B = {B:6.2f} km/s/kpc;  '
          f'checks: A-B-Omega = {A - B - Om_s:.1e}, kappa^2 + 4B(A-B) = {ka_s**2 + 4 * B * (A - B):.1e}')
    assert abs(A - B - Om_s) < 1e-6 and abs(ka_s**2 + 4 * B * (A - B)) < 1e-3 * ka_s**2
print('Gaia DR1 (Bovy 2017):            A = 15.30, B = -11.90 km/s/kpc  (+- 0.4)')

Both models give $A - B = \Omega \approx 27.5$–$28.4$ km/s/kpc, close to the measured
$15.3 + 11.9 = 27.2$ km/s/kpc. They differ in how $\Omega$ splits between $A$ and $B$, which is set by the
local slope of the rotation curve, $dv_c/dR = -(A + B)$:

- MWPotential2014 has a gently falling curve at the Sun and matches the Gaia values within about
  $1\sigma$;
- McMillan (2017) is locally flat ($A \approx -B$), so its $B$ is about 2 km/s/kpc too negative.

The Gaia measurement implies $dv_c/dR = -3.4 \pm 0.6$ km/s/kpc locally, steeper than the
$-1.7$ km/s/kpc that Eilers et al. (2019) find on average over 5–25 kpc. Local measurements are affected
by non-axisymmetric motions: Bovy also measured the constants $C$ and $K$, which vanish in an
axisymmetric galaxy. They do not, a sign of the bar and spiral arms.

## Summary

- In an axisymmetric potential $E$ and $L_z$ are conserved; the motion in the meridional plane is
  confined by the zero-velocity curve $\Phi_{\rm eff}(R, z) = E$.
- Most orbits also conserve a third integral: on a surface of section they trace closed curves.
  It is not the vertical energy; AGAMA's vertical action $J_z$ is a good approximation to it.
- Near-circular orbits are epicycles: radial oscillation at $\kappa$, vertical at $\nu$, rotation at $\Omega$.
  At the Sun $\kappa/\Omega \approx 1.3$ and the epicycle is elongated in the direction of rotation.
- The Oort constants $A$, $B$ measure the local shear and vorticity; $A - B = \Omega$.

| task | AGAMA |
|---|---|
| force and its derivatives | `pot.eval(points, acc=True, der=True)` (columns $xx, yy, zz, xy, yz, zx$) |
| guiding radius for given $L_z$ | `pot.Rcirc(L=Lz)` |
| orbital frequencies $\Omega_r, \Omega_z, \Omega_\phi$ | `agama.ActionFinder(pot)(xv, frequencies=True)` |
| vertical action $J_z$ | `agama.ActionFinder(pot)(xv)[:, 1]` |

## Exercises

1. **Hotter orbits.** Repeat the epicycle test of §5.3 with $v_R$ = 10, 20, 40, 80 km/s. Plot the relative
   error of $\kappa$ as an estimate of the measured radial frequency against $v_R$. Where does the
   approximation fail?
2. **Resonant islands.** In §5.2, launch more orbits with starting radii close to the one that produced
   the chain of arcs. Map the islands, and use `ActionFinder` frequencies to find the ratio
   $\Omega_r/\Omega_z$ of the orbit at their centre. Then repeat §5.2 with $L_z = 0.3\,R_0 v_c$.
3. **Vertical frequency and disk mass.** Near the midplane, $\nu^2 \approx 4\pi G\rho_0$, where $\rho_0$ is the
   midplane density (Poisson's equation for a thin disk). Check this at $R_0$ using `pot.density`.
4. **Oort constants across the Galaxy.** Plot $A(R)$ and $B(R)$ for MWPotential2014 from 4 to 20 kpc.
   Where does the rotation curve rise ($A + B < 0$) and fall?
5. **Solar orbit.** The Sun moves with $(v_R, v_\phi, v_z) \approx (11, 245, 7)$ km/s relative to the Galactic
   centre (signs depend on convention). Integrate its orbit in both models and find its pericentre,
   apocentre and maximum height above the plane.

## References

- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §3.2 and §3.3.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, chapters 8–9: https://galaxiesbook.org
- J. H. Oort (1927), BAN 3, 275: https://ui.adsabs.harvard.edu/abs/1927BAN.....3..275O
- J. Bovy (2017), "Galactic rotation in Gaia DR1", MNRAS 468, L63: https://doi.org/10.1093/mnrasl/slx027
- A.-C. Eilers et al. (2019), ApJ 871, 120: https://doi.org/10.3847/1538-4357/aaf648